In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

cat = ['lead_source','industry','employment_status','location']
num = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
df[cat] = df[cat].fillna('NA')
df[num] = df[num].fillna(0.0)

# Q1
print("Q1", df['industry'].mode()[0])

# Q2
print("Q2\n", df[num].corr())

# Split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
y_train = df_train.converted.values
y_val = df_val.converted.values
df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)

# Q3
for c in cat:
    print("Q3", c, round(mutual_info_score(y_train, df_train[c]), 2))

# Q4
feats = cat + num
def fit_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    Xt = dv.fit_transform(df_train[features].to_dict(orient='records'))
    Xv = dv.transform(df_val[features].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42).fit(Xt, y_train)
    return accuracy_score(y_val, m.predict(Xv))

base = fit_eval(feats)
print("Q4", round(base, 2))

# Q5
for f in ['lead_source','number_of_courses_viewed','interaction_count']:
    acc = fit_eval([x for x in feats if x != f])
    print("Q5", f, base - acc)

# Q6
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print("Q6", C, round(fit_eval(feats, C), 3))

Q1 technology
Q2
                           annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
Q3 lead_source 0.03
Q3 industry 0.0
Q3 employment_status 0.02
Q3 location 0.0
Q4 0.65
Q5 lead_source 0.0030000000000000027
Q5 number_of_courses_viewed 0.0020000000000000018
Q5 interaction_count 0.04400000000000004
Q6 1e-06 0.598
Q6 1e-05 0.598
Q6 0.0001 0.613
Q6 0.001 0.645
